In [190]:
import plotly.graph_objects as go
from dash import Dash, html, dcc
import plotly.express as px
from plotly.subplots import make_subplots

import geopandas as gpd
from shapely.geometry import Point
from geodatasets import get_url
import re

import pandas as pd
import numpy as np

In [191]:
societies = pd.read_csv('../D-PLACE-dplace-cldf-908f302/cldf/societies.csv')

In [192]:
#start gemini

# 2. Convert to a GeoDataFrame
geoData = gpd.GeoDataFrame(
    societies, 
    geometry=gpd.points_from_xy(societies['Longitude'], societies['Latitude']),
    crs="EPSG:4326"
)

In [193]:
# 3. Load the world map directly from the web
# Using the 110m resolution "admin_0_countries" map
world_url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(world_url)

# 4. Filter to just what we need to keep it fast
world = world[['CONTINENT', 'geometry']]

In [194]:
# 5. Spatial Join
# 'predicate="within"' checks if the point is inside the continent polygon
result = gpd.sjoin(geoData, world, how="left", predicate="within")
# End Gemini

In [195]:
#just to see if there are any other "types"\
type_group = result.groupby('type').count()['ID']
type_group

type
languoid    4085
society     2599
Name: ID, dtype: int64

In [196]:
# I'm going to just focus on socieities
result_soc = result[result['type'] == 'society']
result_soc.shape

(2599, 23)

In [197]:
#Let's make an easier dataframe
SocDF = result_soc.drop(columns=['Glottocode', 'xd_id', 'HRAF_name_ID', 'comment', 'glottocode_comment', 'Language_Level_Glottocodes', 'ISO639P3code', 'Contribution_ID', 'type', 'HRAF_ID', 'origLat', 'origLong', 'geometry', 'index_right'])
#i removed 13 columns that I either didn't need/want or were redundant

In [198]:
SocDF['main_focal_year'] = pd.to_numeric(SocDF['main_focal_year'])

In [199]:
americamap = {
    "North America": "Americas",
    "South America": "Americas",
    "Africa": "Africa",
    "Asia": "Asia",
    "Europe": "Europe",
    "Oceania": "Oceania"
}

SocDF['Continent2'] = SocDF['CONTINENT'].map(americamap)
SocDF.tail()

,ID,Name,Latitude,Longitude,Name_and_ID_in_source,alt_names_by_society,main_focal_year,region,CONTINENT,Continent2
2594,WNAI168,Santa Clara,36.0,-106.3,Santa Clara Tewa (J168),Santa Clara Tewa,NaN,South-Central U.S.A.,North America,Americas
2595,WNAI169,Nambé Pueblo,35.9,-105.9,Nambe Tewa (J169),Nambe Tewa,NaN,South-Central U.S.A.,North America,Americas
2596,WNAI170,Taos,36.5,-105.5,Taos (J170),NaN,NaN,South-Central U.S.A.,North America,Americas
2597,WNAI171,Isleta,34.8,-106.7,Isleta (J171),NaN,NaN,South-Central U.S.A.,North America,Americas
2598,WNAI172,Jemez,35.8,-106.8,Jemez (J172),Walatowa; Towa,NaN,South-Central U.S.A.,North America,Americas


In [200]:
continent = SocDF.groupby('Continent2').count()['ID']

In [201]:
year_group = SocDF.groupby('main_focal_year').count()['ID']

In [202]:
worldepic = pd.read_csv('world-folk-epics-longlat.csv')

In [203]:
epicdata = pd.DataFrame(worldepic)

In [204]:
region_group = epicdata.groupby('Region').count()['Title']

In [205]:
regionmap1 = {
    "African": "Africa",
    "Albanian": "Europe",
    "American": "Americas",
    "Balto–Slavic": "Europe",
    "Celtic": "Europe",
    "East and Central Asian": "Asia",
    "Germanic": "Europe",
    "Greek": "Europe",
    "Italic/Romance": "Europe",
    "Northeast Caucasian": "Asia/Europe",
    "South Asian (Indian Subcontinent)": "Asia",
    "Southeast Asian": "Asia",
    "Southwest Asian (Arabian Peninsula)": "Asia",
    "Uralic": "Asia/Europe"
}

In [206]:
regionmap = {
    "African": "Africa",
    "Albanian": "Europe",
    "American": "Americas",
    "Balto–Slavic": "Europe",
    "Celtic": "Europe",
    "East and Central Asian": "Asia",
    "Germanic": "Europe",
    "Greek": "Europe",
    "Italic/Romance": "Europe",
    "Northeast Caucasian": "Asia",
    "South Asian (Indian Subcontinent)": "Asia",
    "Southeast Asian": "Asia",
    "Southwest Asian (Arabian Peninsula)": "Asia",
    "Uralic": "Europe"
}

In [207]:
epicdata['Continent'] = epicdata['Region'].map(regionmap)

epicdata.head()

,Title,Description,Language,Tradition,Region,Year (Rough),Lat,Long,Continent
0,T'heydinn,"a Mauritanian epic ensemble, performed in Has...",Hassaniya,Mauritanian,African,1600.0,20.0,-12.0,Africa
1,Gassire's Lute,"an epic from the Soninke people, West Africa",Soninke,Soninke,African,400.0,15.0,-8.0,Africa
2,Bayajidda,a West African epic,Hausa,Hausa,African,900.0,12.5,7.5,Africa
3,Eri,a West African epic,Igbo,Igbo,African,900.0,6.0,7.0,Africa
4,Oduduwa,a West African epic,Yoruba,Yoruba,African,1000.0,7.5,4.5,Africa


In [208]:
# I want the year to say BCE or CE
conditions = [epicdata["Year (Rough)"] < 0]
choices = ["BCE"]
epicdata['Yr Unit'] = np.select(conditions, choices, default="CE")

epicdata.tail()

,Title,Description,Language,Tradition,Region,Year (Rough),Lat,Long,Continent,Yr Unit
113,Hinilawod,"an epic of the Panay-Bukidnons of Panay, the...",Hiligaynon/Kinaray-a,Panay-Bukidnon,Southeast Asian,1550.0,11.0,122.5,Asia,CE
114,Darangen,an epic of the Maranao of Mindanao,Maranao,Maranao,Southeast Asian,1300.0,8.0,124.3,Asia,CE
115,Bharatayuddha,the longest epic in the world,Old Javanese,Javanese,Southeast Asian,1157.0,-7.5,112.5,Asia,CE
116,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CE
117,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CE


In [209]:
# I want the year to say BCE or CE
conditions = [SocDF["main_focal_year"] < 0, SocDF["main_focal_year"] >= 0]
choices = ["BCE", "CE"]
SocDF['Yr Unit'] = np.select(conditions, choices, default="")

SocDF.tail()

,ID,Name,Latitude,Longitude,Name_and_ID_in_source,alt_names_by_society,main_focal_year,region,CONTINENT,Continent2,Yr Unit
2594,WNAI168,Santa Clara,36.0,-106.3,Santa Clara Tewa (J168),Santa Clara Tewa,NaN,South-Central U.S.A.,North America,Americas,
2595,WNAI169,Nambé Pueblo,35.9,-105.9,Nambe Tewa (J169),Nambe Tewa,NaN,South-Central U.S.A.,North America,Americas,
2596,WNAI170,Taos,36.5,-105.5,Taos (J170),NaN,NaN,South-Central U.S.A.,North America,Americas,
2597,WNAI171,Isleta,34.8,-106.7,Isleta (J171),NaN,NaN,South-Central U.S.A.,North America,Americas,
2598,WNAI172,Jemez,35.8,-106.8,Jemez (J172),Walatowa; Towa,NaN,South-Central U.S.A.,North America,Americas,


In [210]:
epicscontinent = epicdata.groupby('Continent').count()["Title"]
epicscontinent

Continent
Africa      11
Americas     2
Asia        67
Europe      28
Name: Title, dtype: int64

In [211]:
#Gemini did this map from the loc_list.txt file I exported for it, I would have taken years honestly

#Also I still had to edit this, I noticed "Wales" was a dropped row originally, among others, sigh
region_map = {
    'African': [
        'African', 'Sub-Saharan', 'Nigeria', 'Kenya', 'Zulu', 'Bantu', 'Yoruba', 'Sudan',
        'Congo', 'Ethiopia', 'Somali', 'Mali', 'Ghana', 'Senegal', 'Tanzania', 'Madagascar'
    ],
    'Albanian': ['Albanian', 'Albania'],

    'American': [
        'American', 'North American', 'Native American', 'Inuit', 'Navajo', 'Cherokee',
        'Aztec', 'Maya', 'Inca', 'Kechua', 'Iroquois', 'Sioux', 'Apache', 'Brazil', 'Mexico', 
        'Canada', 'Jamaica', 'Antigua', "Zapotec", "Quiche"
    ],
    'Balto-Slavic': [
        'Russian Federation', 'Ukraine', 'Ukrainian', 'Belarus', 'Poland', 'Polish',
        'Czech', 'Slovak', 'Bulgaria', 'Bulgarian', 'Serbian', 'Croatian', 'Slovenian',
        'Latvia', 'Lithuania', 'Estonia', 'Slavic', 'Northern Ukrainians', 'Serbia', 'Moravia', 'Kashubians', 'Rusyns'
    ],
    'Celtic': [
    'Celtic', 
    'Wales', 'Welsh', 
    'Ireland', 'Irish', 'Gaelic', 
    'Scotland', 'Scottish', 
    'Brittany', 'Breton', 
    'Cornwall', 'Cornish', 
    'Isle of Man', 'Manx'],

    
    'East and Central Asian': [
        'Chinese', 'Japanese', 'Korean', 'Mongolian', 'Tibetan', 'Manchu', 'Taiwan', 'East Asian', 'Japan', 'China', 'Mongols'
    ],
    'Germanic': [
        'Germany', 'German', 'Iceland', 'Norway', 'Norwegian', 'Sweden', 'Swedish', 'Switzerland', 'Luxemborg',
        'Denmark', 'Danish', 'Dutch', 'Netherlands', 'Austria', 'English', 'Schleswig-Holstein',
        'Saxony', 'Bavaria', 'Lower Saxony', 'England', 'Scandinavia', 'Shetland Islands', 'Flanders', "Grimm",
        'Bartsch', 'Schoppner', 'Lyncker', "Kuhn", "Haas"
    ],
    'Greek': ['Greece', 'Greek', 'Hellenic', 'Cyprus'],

    'Italic/Romance': [
        'Italy', 'Italian', 'France', 'French', 'Spain', 'Spanish', 'Portugal', 'Portuguese',
        'Romania', 'Romance', 'Latin', 'Toscana', 'Sicily', 'Umbria', 'Marche', 'Lazio', "Perrault", "Basile"
    ],
    'Kartvelian / South Caucasian': [
        'Georgia', 'Georgian', 'Mingrelian', 'Svan', 'Laz', 'Kartvelian'
    ],
    'Levantine & North African': [
        'Levant', 'Syria', 'Lebanon', 'Palestine', 'Jordan', 'Egypt', 'Morocco', 'Tunisia',
        'Algeria', 'Libya', 'Maghreb', 'Berber'
    ],
    'Northeast Caucasian': [
        'Chechen', 'Ingush', 'Dagestan', 'Avar', 'Lezgin', 'Circassian', 'Armenians'
    ],
    'Oceanian / Pacific Islander': [
        'Oceania', 'Polynesia', 'Melanesia', 'Micronesia', 'Hawaii', 'Maori', 'Fiji',
        'Samoa', 'Tonga', 'Tuamotu', 'Gilbert Islands', 'Solomons'
    ],
    'South Asian': [
        'Indian', 'Vedic', 'Brahman', 'Hindu', 'Pakistan', 'Bengali', 'Tamil', 'Punjabi',
        'Sri Lanka', 'Nepal', 'Bangladesh', 'Panchtantra', 'Jatakas', 'Indian Literary'
    ],
    'Southeast Asian': [
        'Vietnam', 'Thailand', 'Thai', 'Indonesia', 'Malay', 'Philippines', 'Burma',
        'Myanmar', 'Cambodia', 'Laos', 'Singapore'
    ],
    'Southwest Asian': [
        'Arabian', 'Saudi', 'Yemen', 'Oman', 'Qatar', 'Kuwait', 'Emirates', 'Bedouin', 'Bedouins'
    ],
    'Turkic': [
        'Turkish', 'Turkey', 'Azerbaijan', 'Kazakh', 'Kirghiz', 'Uzbek', 'Turkmen',
        'Tatar', 'Bashkir', 'Uygur', 'Yakut', 'Tuva'
    ],
    'Uralic': [
        'Finnish', 'Finland', 'Hungarian', 'Hungary', 'Estonian', 'Sami', 'Uralic', 'Nenets'
    ],
    'West Asian / Indo-Iranian': [
        'Persian', 'Iran', 'Tajik', 'Kurdish', 'Baluchi', 'Pashto', 'Afghan', 'Persia', 'Armenia', 'Kurds',
        "Hartland", "Temme"
    ]
}

In [212]:
def process_folklore_data(filepath):
    # Load the data
    df = pd.read_csv(filepath)
    
    # Clean the column name (fixing the space issue: 'Location / Tradition')
    target_col = 'Location / Tradition'
    
    # 2. Assignment Logic
    def assign_region(loc_string):
        if pd.isna(loc_string):
            return 'Unknown'
        for region, keywords in region_map.items():
            if any(key.lower() in str(loc_string).lower() for key in keywords):
                #i love the above line! if it fits one word, it puts the region in, saves tons!
                return region
        return 'Other / Unclassified'

    # Create the new column
    df['Region_Group'] = df[target_col].apply(assign_region)
    
    # 3. Convert to GeoDataFrame (using your existing Lat/Long)
    # Removing rows without coordinates for cleaner mapping
    df = df.dropna(subset=['Latitude', 'Longitude'])
    geometry = [Point(xy) for xy in zip(df['Longitude'], df['Latitude'])]
    gdf = gpd.GeoDataFrame(df, crs="EPSG:4326", geometry=geometry)
    
    return gdf

# Apply it to your magic file
eternity = process_folklore_data('eternal-c.csv')
warning  = process_folklore_data('warnings-c.csv')
betray = process_folklore_data('betray-c.csv')
magic = process_folklore_data('magic-c.csv')
shapeshift = process_folklore_data('changeling-c.csv')
# eternity.tail()
#great let's get these on a map

In [213]:
#I asked Gemini to help with all the undefined coordinates in motifs, didn't work and I gave up on it
def spatial_classification(row):
    # Only fix the ones that are currently unclassified
    if row['Region_Group'] != 'Other/Unclassified':
        return row['Region_Group']
    
    lat, lon = row['Latitude'], row['Longitude']
    
    # Simple Polygon Logic (Approximate Bounding Boxes)
    if 35 <= lat <= 72 and -25 <= lon <= 45:
        return 'Europe (Spatial)'
    if -35 <= lat <= 35 and -20 <= lon <= 50:
        return 'Africa (Spatial)'
    if 15 <= lat <= 75 and -170 <= lon <= -50:
        return 'Americas (Spatial)'
    if -10 <= lat <= 80 and 50 <= lon <= 180:
        return 'Asia & Oceania (Spatial)'
        
    return 'Other/Still Undefined'

# Apply this to your 'magic' dataframe
magic['Region_Group'] = magic.apply(spatial_classification, axis=1)
#WHY IS NOTHING HAPPENING ;_; ugh

# with pd.option_context('display.max_rows', None):
#     print(magic['Region_Group'])

In [214]:
#A traditions vs. epics comparison
fig5 = make_subplots(rows = 1, cols = 2, specs=[[{'type':'domain'}, {'type':'domain'}]])

colors = px.colors.qualitative.Bold # Or Light24, Bold, etc.

fig5.add_trace(
    go.Pie(
    labels = continent.index,
    values=continent.values,
    name="Traditions by Continent"),
    row=1, col=1
    )
#I should write something short about what a tradition is, how it's essentially a culture or a larger idea of a tribe, a community

fig5.add_trace(
    go.Pie(
    labels = epicscontinent.index,
    values=epicscontinent.values,
    name="World Folk Epics by Continent"),
    row=1, col=2
    )

fig5.update_layout(title={
        # 'text': "Traditions Vs. Recognized World Folk Epics by Continent", 
        'x':0.5,
        'xanchor': "center"}, 
                   annotations=[dict(text='Traditions', x=sum(fig5.get_subplot(1, 1).x) / 2, y=0.5,
                      font_size=20, showarrow=False, xanchor="center"),
                 dict(text='World Folk<br>Epics', x=sum(fig5.get_subplot(1, 2).x) / 2, y=0.5,
                      font_size=18, showarrow=False, xanchor="center")])

fig5.update_traces(hole=.4, hoverinfo='label+percent', textinfo='value', textfont_size=15,
                  marker=dict(colors=colors))
fig5.show()
#I chose to group North and South America so the comparison would be better
#I also edited the mapping so that Uralic is European and NE Caucasia is Asian so there wouldn't be a Asian/European category

#There are probably better ways to compare I tried a geo Bubble map and I simply could NOT get it to work 
#i used ISO-3 instead of log/lat and made a new column with their respective ISO-3 continent coordinates but it wouldnt work
#sometimes :) you have to tell yourself "it's good enough, go to bed"

In [215]:
year_epics = epicdata.groupby('Year (Rough)').count()["Title"]

In [216]:
SocDF['main_focal_year'] = pd.to_numeric(SocDF['main_focal_year'])

In [217]:
all_regions = sorted(SocDF['region'].unique())

# Group A: Everything with a valid year (for the animation)
animated_df = SocDF[SocDF['main_focal_year'].notna()]

# Group B: Everything missing a year (the static background)
na_df = SocDF[SocDF['main_focal_year'].isna()]

#i originally had this set min, max, and a 500 step count to a list using numpy arrange
# but honestly this data is more exponential so I just made it myself
years = [0, 500, 1000, 1200, 1400, 1600, 1700, 1800, 1850, 1900, 1990]

colors = px.colors.qualitative.Light24 # Or Light24, Bold, etc.

color_map = {region: colors[i % len(colors)] for i, region in enumerate(all_regions)}
#I'm doing this bit above in order to have each region maintain a unique color

layout = go.Layout(
    #title="Traditions Globally by Focal Year", 
                   legend_title="region")

fig4 = go.Figure(layout=layout)

frame_data = []

# 1. Add Initial Traces
for region in animated_df['region'].unique():
    # Show only data from before the year 0
    mask = (animated_df['region'] == region) & (animated_df['main_focal_year'] <= 1999)
    sub = animated_df[mask]

    # I want NAs to be there the whole time
    sub_na = na_df[na_df['region'] == region]
    
    fig4.add_trace(go.Scattergeo(
        lat=sub_na["Latitude"],
        lon=sub_na["Longitude"],
        text=sub_na["Name"],
        name=f"{region} (Year Unknown)",
        customdata=sub_na["main_focal_year"],
        hovertemplate="<b>%{text}</b><br>%{customdata}",
        mode='markers',
        marker=dict(
            size=4, 
            opacity=0.5,
            color="lightgray",
            symbol="diamond" #gray/diamonds to set them apart
        ),
        showlegend=False
    ))
    fig4.add_trace(go.Scattergeo(
        lat=sub["Latitude"],
        lon=sub["Longitude"],
        text=sub["Name"],
        customdata=sub[["main_focal_year", "Yr Unit"]],
        name=region,
        hovertemplate="<b>%{text}</b> <br>%{customdata[0]} %{customdata[1]}",
        mode='markers',
        marker=dict(size=5, opacity=.2, symbol='circle', color=color_map[region]),
        showlegend=False
        #These showed up twice unless I set the first frame to false
    ))

# 2. Create the Frames (One for each 'step' in time)
frames = []
for yr in years:
    frame_data = []

    # FIRST: Add the 'Static' NA traces for every frame
    for region in all_regions:
        sub_na = na_df[na_df['region'] == region]
        frame_data.append(go.Scattergeo(
            lat=sub_na["Latitude"],
            lon=sub_na["Longitude"],
            text=sub_na["Name"],
            name=f"{region} (Year Unknown)",
            customdata=sub_na["main_focal_year"],
            hovertemplate="<b>%{text}</b><br>%{customdata}",
            mode='markers',
            marker=dict(
                size=4, 
                opacity=0.5,
                color="lightgray",
                symbol="diamond"
            ),
            showlegend=False
    ))
    
    for region in animated_df['region'].unique():
        # Cumulative: Show everything up to the current slider year
        mask = (animated_df['region'] == region) & (animated_df['main_focal_year'] <= yr)
        sub = animated_df[mask]
        
        frame_data.append(go.Scattergeo(
            lat=sub["Latitude"],
        lon=sub["Longitude"],
        text=sub["Name"],
        customdata=sub[["main_focal_year", "Yr Unit"]],
        name=region,
        hovertemplate="<b>%{text}</b><br>%{customdata[0]} %{customdata[1]}",
        mode='markers',
        marker=dict(size=5, opacity=.2, symbol='circle', color=color_map[region]),
        showlegend=True
        ))
    frames.append(go.Frame(data=frame_data, name=str(yr)))

fig4.frames = frames

fig4.update_layout(
    # title="Traditions Globally by Focal Year",
    showlegend=True,
    geo=dict(
        projection_type='natural earth',
        showcoastlines=False,
        showland=True, landcolor="DarkSeaGreen",
        showocean=True, oceancolor="Azure",
    ),
    updatemenus=[{
        "type": "buttons",
        "buttons": [{
            "label": "Play",
            "method": "animate",
            "args": [None, {"frame": {"duration": 500, "redraw": True}, "fromcurrent": True}]
        }]
    }],
    sliders=[{
        "active": 0,
        "steps": [{
            "label": str(yr),
            "method": "animate",
            "args": [[str(yr)], {
                "frame": {"duration": 100, "redraw": True}, 
                "fromcurrent": True,
                "transition": {"duration": 500},
                "mode": "immediate"}]
        } for yr in years]
    }]
)
#I think I'm going to get rid of the play button, and also try to fix the timeline scroll (bugs out if scrolled to the end)

#Update: did not get rid of the play button, 
# but I did make both timelines start at their endpoint so if someone didn't interact they'd still get good information
fig4.show()

In [218]:
fig2 = make_subplots(rows = 2, cols = 2,
    specs=[[{'type':'scattergeo'}, {'type':'scattergeo'}],
           [{'type':'scattergeo'}, {'type':'scattergeo'}]],
           subplot_titles=("Eternal Punishments/Tasks", "Moral Heedings/Warnings", "Betrayals", "Magic/Sorcery"))

regions = sorted(magic['Region_Group'].unique())

colors = px.colors.qualitative.Light24 # Or Light24, Bold, etc.

color_map = {region: colors[i % len(colors)] for i, region in enumerate(regions)}
#I'm doing this bit above in order to have each region maintain a unique color

fig2.update_layout(
    #title="Folklore Motifs by Category", 
                   legend_title="Region", showlegend=False)

# Loop through each unique Region to create separate traces
for r in regions:
    eternity_sub = eternity[eternity['Region_Group'] == r]
    #eternity
    fig2.add_trace(go.Scattergeo(
                lat=eternity_sub["Latitude"],
                lon=eternity_sub["Longitude"],
                name=r,
                mode='markers',
                legendgroup="group1",

                marker=dict(
                    size=7,
                    opacity=0.3,
                    color=color_map[r])
    ),row=1, col=1)
    
    #warnings
    warning_sub = warning[warning['Region_Group'] == r]
    #eternity
    fig2.add_trace(go.Scattergeo(
                lat=warning_sub["Latitude"],
                lon=warning_sub["Longitude"],
                name=r,
                mode='markers',
                legendgroup="group1",
                showlegend=False,
                marker=dict(
                    size=7,
                    opacity=0.3,
                    color=color_map[r])
    ),row=1, col=2)

    #betray
    betray_sub = betray[betray['Region_Group'] == r]
    fig2.add_trace(go.Scattergeo(
                lat=betray_sub["Latitude"],
                lon=betray_sub["Longitude"],
                name=r,
                mode='markers',
                legendgroup="group1",
                showlegend=False,
                marker=dict(
                    size=7,
                    opacity=0.3,
                    color=color_map[r])
    ),row=2, col=1)
    #magic
    magic_sub = magic[magic['Region_Group'] == r]
    fig2.add_trace(go.Scattergeo(
                lat=magic_sub["Latitude"],
                lon=magic_sub["Longitude"],
                name=r,
                mode='markers',
                legendgroup="group1",
                showlegend=False,
                marker=dict(
                    size=7,
                    opacity=0.3,
                    color=color_map[r])
    ),row=2, col=2)
    
    #i removed shapeshifting, doesn't aid the overall narrative

fig2.update_geos(projection_type="natural earth",
        showcoastlines=False,
        showland=True, landcolor="DarkSeaGreen",
        showocean=True, oceancolor="Azure",)

# 1. Calculate counts for each group
counts = {
    "eternity": len(eternity),
    "warning": len(warning),
    "betray": len(betray),
    "magic": len(magic)
}

# 2. Add annotations (x and y are from 0 to 1 across the whole figure)
# Adjust the 'y' values slightly if they overlap with your map borders
fig2.add_annotation(dict(x=0.2, y=0.55, text=f"{counts['eternity']} data points", 
                         showarrow=False, xref="paper", yref="paper", font=dict(size=12)))
fig2.add_annotation(dict(x=0.8, y=0.55, text=f"{counts['warning']} data points", 
                         showarrow=False, xref="paper", yref="paper", font=dict(size=12)))
fig2.add_annotation(dict(x=0.2, y=-0.1, text=f"{counts['betray']} data points", 
                         showarrow=False, xref="paper", yref="paper", font=dict(size=12)))
fig2.add_annotation(dict(x=0.8, y=-0.1, text=f"{counts['magic']} data points", 
                         showarrow=False, xref="paper", yref="paper", font=dict(size=12)))

# 3. Increase bottom margin to make room for the bottom labels
fig2.update_layout(margin=dict(b=50), showlegend=False)
#you cant do anything with the legend, everything disappears if you click it, might as well make the graph bigger by chucking it

fig2.show()

In [219]:
len(years)

11

In [220]:
regions = epicdata.groupby('Region').count()["Title"]

regions

Region
African                                11
Albanian                                1
American                                2
Balto-Slavic                            8
Celtic                                  8
East and Central Asian                 14
Germanic                                8
Greek                                   3
Italic/Romance                          6
Northeast Caucasian                     1
South Asian (Indian Subcontinent)      18
Southeast Asian                        13
Southwest Asian (Arabian Peninsula)    21
Uralic                                  2
Name: Title, dtype: int64

In [221]:
regions = epicdata['Region'].unique()

In [222]:
colors = px.colors.qualitative.Light24 # Or Light24, Bold, etc.

color_map = {region: colors[i % len(colors)] for i, region in enumerate(regions)}
#Gemini helped with color map

# years = sorted(epicdata['Year (Rough)'].unique())
years = [-5500, -2500, -1000, 0, 500, 1000, 1250, 1500, 1750, 1990]

# I was suffering w the timeline code so I gave it to Gemini, it did the initial but both of my timelines look nothing like where they began so I can't really give more info
fig1 = go.Figure()

frame_data = []
# 1. Add Initial Traces (The starting point of the animation)
first_year = years[0]
last_year = years[9]
for region in epicdata['Region'].unique():
    # Show only data from the very first available year
    mask = (epicdata['Region'] == region) & (epicdata['Year (Rough)'] <= last_year)
    sub = epicdata[mask]
    
    fig1.add_trace(go.Scattergeo(
        lat=sub["Lat"],
        lon=sub["Long"],
        text=sub["Title"],
        customdata=sub[["Description", "Year (Rough)", "Yr Unit"]],
        name=region,
        hovertemplate="<b>%{text}</b> %{customdata[1]} %{customdata[2]}<br />%{customdata[0]}",
        # finally figured out hot to have more than two hover datum here https://plotly.com/python/hover-text-and-formatting/
        mode='markers',
        marker=dict(size=10, opacity=.5, color=color_map[region])
    ))

# 2. Create the Frames (One for each 'step' in time)
frames = []
for yr in years:
    frame_data = []
    for region in epicdata['Region'].unique():
        # Cumulative: Show everything up to the current slider year
        mask = (epicdata['Region'] == region) & (epicdata['Year (Rough)'] <= yr)
        sub = epicdata[mask]
        
        frame_data.append(go.Scattergeo(
            lat=sub["Lat"],
            lon=sub["Long"],
            text=sub["Title"],
            customdata=sub[["Description", "Year (Rough)", "Yr Unit"]],
            name=region,
            hovertemplate="<b>%{text}</b> %{customdata[1]} %{customdata[2]}<br />%{customdata[0]}",
            mode='markers',
            marker=dict(size=10, opacity=.5, color=color_map[region])
            ))
    frames.append(go.Frame(data=frame_data, name=str(yr)))

fig1.frames = frames

fig1.update_layout(
    # title="The Spread of World Folk Epics Over Time",
    #took out title because it's bigger if I use an html H tag
    showlegend=True,
    geo=dict(
        projection_type='natural earth',
        showcoastlines=False,
        showland=True, landcolor="DarkSeaGreen",
        showocean=True, oceancolor="Azure",
    ),
    updatemenus=[{
        "type": "buttons",
        "buttons": [{
            "label": "Play",
            "method": "animate",
            "args": [None, {"frame": {"duration": 500, "redraw": True}, "fromcurrent": True}]
        }]
    }],
    sliders=[{
        "active": 0,
        "steps": [{
            "label": str(yr),
            "method": "animate",
            "args": [[str(yr)], {
                "frame": {"duration": 100, "redraw": True}, 
                "fromcurrent": True,
                "transition": {"duration": 500},
                "mode": "immediate"}]
        } for yr in years]
    }]
)
fig1.show()

In [223]:
# fun fact! i learned native dash apps automatically look in their folder for assets/ and then read all those css files before the styling in app.
# So I style using that :) /assets/style.css
app = Dash(__name__)

app.layout = html.Div([
    html.Div([
    html.H1("Humans, Stories, & Time"),
    html.P("This app was created as my final interactive data project, exploring the oral tradition in a new context. I hope that seeing qualitative data quantitatively, we can see different aspects we hadn't previously."
           ),
    html.P("Spanning millennia, I want to see where the stories we have come from, and what themes, lessons, and truths people wanted to impart with their storytelling."),
    html.P(["Since the beginning of humanity, stories have been a way that we teach our communities and young what traits our societies value, and what traits are hated or less favorable.", html.B(" The oral tradition is the oldest surviving evidence we have for pedagogy, the practice of teaching,")," and on a neurological level it is closer to experiential learning than explicit recall, what most textbooks and exams are based on. Hopefully, though showing the importance of storytelling to the ongoing human narrative, its breadth and connections over time, space, and cultures can be highlighted and appreciated moving forward not just as fun, but essential."]),
    ]),
    html.H2("Traditions Vs. Recognized World Folk Epics by Continent"),
    html.Div([
        dcc.Graph(id="pie-trad-vs-wfepic", figure=fig5, className='flop'),
        html.Div([
            html.P([ html.B("A Tradition is cultural continuity in social attitudes, customs, and institutions."), " When speaking anthropologically, a Tradition can be thought of like a culture or society at a given time."]),  
            html.P([ html.B("Epics are long, sprawling narrative stories."), " As storytelling is a ubiquitous human trait, every culture has stories and epics. World folk-epics are defined as 'not just literary masterpieces but also an integral part of the worldview of a people.' They were originally oral literatures, which were later written down by either single author or several writers."]),  
            html.P("Through exploring this graph, you may see a disparity in the proportions of people on a given continent to what stories of theirs are regarded as 'world folk epics'.")      
        ], className='flip')
    ], className='grid'),

    html.H2("Folklore Motifs by Category"),
    html.Div([
        html.Div([
            html.H4("Here, four common motifs are presented:"),
            html.Ul([
                html.Li(["Eternal punishment ", html.B("(Sisyphus, Greece)")]),
                html.Li(["Moral Warnings ", html.B("(Ali Baba & the Forty Thieves, Arabia)")]),
                html.Li(["Betrayal ", html.B("(The Badger and the Bear, Lakota Tribe)")]),
                html.Li(["Magic ", html.B("(The Magic Paintbrush, China)")])
        ]),
            html.P("See if there any areas you notice that have notably more or less of a certain motif. Often, the motifs and stories are tied to the values of a particular culture, and, through more exploration, there may be stories left to uncover here."),
        ], className='flip'),
        dcc.Graph(id="compare-motifs", figure=fig2, className='flop'),
    ], className='grid'),

    html.H2("Traditions Globally by Focal Year"),
    html.Div([
        
        dcc.Graph(id="timeline-trad", figure=fig4, className='flop'),
        html.Div([
            html.P(["In anthropology, a ", html.B("'focal year'"), " (or more broadly, 'time in the field') refers to ", html.B("the intensive period of participant observation—often lasting a year or more—where an anthropologist immerses themselves in a community to study its culture and social structure"), html.I(" (The Encyclopedia of Anthropology)"), "."]),
            html.P(["For this graph, it's a shorthand for ", html.B("people paid attention."), " It's not the most accurate shorthand for this, but hopefully, seeing where and when 'Western culture' takes note of things elicits some deeper questions."]),
        ], className='flip')
    ], className='grid'),
    html.H2("World Folk Epics Over Time"),
    html.Div([
        
        dcc.Graph(id="timeline-wfepic", figure=fig1, className='flop'),
        html.Div([
            html.P(["You may be familiar with some of these titles such as ", html.I("The One Thousand and One Nights"), " (containing Aladdin), ", html.I("Beowulf"), ", ", html.I("Le Morte d'Arthur"), " (The Legend of King Arthur), ", html.I("the Tanakh"), " (Bible), ", html.I("the Mahabharata"), ", and more. "]),
            html.P(["Many of these stories were developed over centuries or millennia, having little hints and pieces of the history they carry within them. Because of this, for many of these there is no real 'creation date' but through the research of some very intelligent scholars using language, cultural references, and very scientific 'guesstimating' the timeline here was put together. Many of the tales from 1500 CE or later have the year that they were imperialized ", html.I("(or shortly therafter)")," by Western civilization as the tale's origin because that it the earliest date the West can be confident in."]),
        ], className='flip')
    ], className='grid'),

    html.Div([
        html.H4("Disclaimer"),
        html.P("The people are real, the stories are real. The timelines and locations are very approximate, and therefore the story the data tells is skewed. Skewed by who wrote down the history that has been stored, skewed by the biases of everyone who's written down, translated, logged, or shared any of this information, and skewed by me, not by intention, but by my humanity."),
        html.P(["This is inherently qualitative data shown quantitatively, in order to give it a new context. What it isn't is 'accurate'. I cannot find every single myth or tale with moral warnings, eternal tasks, or whatever else. I am only one design and technology student, and even scholars who dedicate their lives to this research would know that a comprehensive and accurate depiction of the global oral tradition of humanity is impossible."]),
        html.Ul([
            html.Li("I used the earliest dates I could find with evidentiary support. Many of these from 1500+ CE are much older, but scholars have not agreed upon even loose dates for their oral creation."),
            html.Li(["Timelines for oral tradition are inherently difficult because",
                html.Ol([html.Li("Word of mouth does not create a 'paper trail', and"),
                         html.Li("Many stories have edits and addendums to make them more pertinent for the times, which means parts of the stories may have references to events thousands of years ago, whereas others could have only been added in the semi recent past.")
                              ])])
            ]),
        html.H6("Bibliography"),
        html.Ul([
            html.Li(["Traditions - from ", html.A("The Database of Places, Language, Culture, and Environment", href="https://d-place.org/") ]),
            html.Li(["Folklore Motif Comparison - from ", html.A("The Folklore Database", href="https://www.folkloredatabase.com"), ". Searches include:",
                     html.Ul([
                     html.Li([html.B("Eternal Tasks: "),"(repetitive OR continuous OR eternity OR eternal) (task OR punishment OR labor) OR (reward OR sacrifice)"]),
                     html.Li([html.B("Moral Warnings/Heedings: "),"(warn* OR caution OR 'do not') AND (greed or pride or hubris or hum* or lie*)"]),
                     html.Li([html.B("Magic/Sorcery: "),"magic* OR enchant* OR sorcer* OR witch* OR supernatural"]),
                     html.Li([html.B("Betrayal: "),"betray* AND ((lover OR lovers OR married OR wife) AND (infidelity OR cheating OR unfaithful)) OR ((broken OR break*) AND TRUST) OR (lie* OR lying)"]),
                     ]),
                     ]),
            html.Li(["World Folk Epics - from ",
            html.Ul([ 
                html.Li([ html.A("Wikipedia: List of World Folk Epics", href="https://en.wikipedia.org/wiki/List_of_world_folk-epics") ]),
                html.Li(["UNESCO's ", html.A("Lists of Intangible Cultural Heritage", href="https://ich.unesco.org/en/lists") ]),
        ]),
                html.Li(["Google Gemini was used for various code snippets (which are noted in the Jupyter Notebook file as they come), and approximate locations for the World Folk Epics."])
          ]) ])
        ], id="disclaimer")
])

if __name__ == '__main__':
    app.run_server(debug=True, use_reloader=False)